# Notebook 01 — Đọc & Làm sạch Dữ liệu
**Nhóm 67 | Tuần 2**

Mục tiêu: Đọc MBPP subset, kiểm tra chất lượng, làm sạch và xuất dữ liệu đã xử lý.

> Chạy notebook 00 trước khi chạy notebook này.

In [ ]:
import json, os, pandas as pd

BASE = '/content/nhom67'
DATA_FILE = f'{BASE}/data/raw/mbpp_subset.json'

with open(DATA_FILE, encoding='utf-8') as f:
    data = json.load(f)

print(f'✓ Đọc xong: {len(data)} bài')
print(f'  Khóa của mỗi bài: {list(data[0].keys())}')

## 1. Bảng thống kê dữ liệu

In [ ]:
from collections import Counter

topics   = Counter(d['topic'] for d in data)
pub_cnts = [len(d['public_tests']) for d in data]
hid_cnts = [len(d['hidden_tests']) for d in data]
desc_len = [len(d['text'].split()) for d in data]

stats = {
    'Tên bộ dữ liệu':          'MBPP (Mostly Basic Python Problems)',
    'Nguồn dữ liệu':           'Google Research (Austin et al., 2021)',
    'Số mẫu (bài toán)':       len(data),
    'Số đặc trưng chính':      '4 (task_id, text, code, test cases)',
    'Loại bài toán':           'Chấm bài lập trình (code grading)',
    'Biến mục tiêu':           'Test Pass Rate (% test vượt qua)',
    'Public test / bài':       f'{min(pub_cnts)}–{max(pub_cnts)} (trung bình {sum(pub_cnts)/len(pub_cnts):.1f})',
    'Hidden test / bài':       f'{min(hid_cnts)}–{max(hid_cnts)} (trung bình {sum(hid_cnts)/len(hid_cnts):.1f})',
    'Độ dài mô tả (từ)':       f'{min(desc_len)}–{max(desc_len)} (trung bình {sum(desc_len)/len(desc_len):.1f})',
    'Chủ đề (topic)':          ', '.join(f'{k}:{v}' for k,v in topics.items()),
}

print('=' * 60)
print('  BẢNG MÔ TẢ BỘ DỮ LIỆU')
print('=' * 60)
for k, v in stats.items():
    print(f'  {k:<30} {v}')
print('=' * 60)

## 2. Làm sạch dữ liệu cơ bản

In [ ]:
import ast

clean_data = []
removed = []

seen_ids = set()

for item in data:
    tid  = item['task_id']
    code = item['code']
    text = item['text'].strip()

    # 1. Kiểm tra trùng lặp task_id
    if tid in seen_ids:
        removed.append((tid, 'Trùng lặp task_id'))
        continue
    seen_ids.add(tid)

    # 2. Kiểm tra mô tả rỗng
    if not text:
        removed.append((tid, 'Mô tả rỗng'))
        continue

    # 3. Kiểm tra syntax code
    try:
        ast.parse(code)
    except SyntaxError as e:
        removed.append((tid, f'Lỗi syntax: {e.msg}'))
        continue

    # 4. Kiểm tra có ít nhất 1 test case
    if len(item.get('public_tests', [])) == 0:
        removed.append((tid, 'Không có test case'))
        continue

    clean_data.append(item)

print(f'Trước làm sạch: {len(data)} bài')
print(f'Sau làm sạch:   {len(clean_data)} bài')
print(f'Đã loại bỏ:     {len(removed)} bài')

if removed:
    print('\nDanh sách bài bị loại:')
    for tid, reason in removed:
        print(f'  task_id {tid}: {reason}')
else:
    print('\n✓ Không có bài nào bị loại — dữ liệu sạch')

In [ ]:
# Lưu dữ liệu đã làm sạch
out_path = f'{BASE}/data/processed/mbpp_clean.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(clean_data, f, ensure_ascii=False, indent=2)

print(f'✓ Đã lưu: {out_path}')
print(f'  Số bài: {len(clean_data)}')

## 3. Phân tích phân bố dữ liệu

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

df = pd.DataFrame([{
    'task_id':    d['task_id'],
    'topic':      d['topic'],
    'desc_words': len(d['text'].split()),
    'pub_tests':  len(d['public_tests']),
    'hid_tests':  len(d['hidden_tests']),
    'code_lines': len([l for l in d['code'].split('\n') if l.strip()]),
} for d in clean_data])

print(df.describe().round(2))
print(f'\nPhân bố chủ đề:')
print(df['topic'].value_counts().to_string())